# [LAB12] 딥러닝 > 신경망의 이해 > 08. 다중선형회귀 + 결과해석

속도에 따른 제동거리 예측 데이터셋

## 📘 #01. 준비작업

### 📝 [1] 패키지 참조

### 📝 [2] 데이터셋 가져오기

In [ ]:
!pip install --upgrade keras-tuner

In [ ]:
from hossam import *
from pandas import DataFrame
from matplotlib import pyplot as plt
import seaborn as sb
import numpy as np
from datetime import datetime as dt
from keras_tuner import Hyperband
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.optimizers import SGD, RMSprop
from tensorflow.keras.losses import mse
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.metrics import RootMeanSquaredError, R2Score
from tqdm.keras import TqdmCallback
from statsmodels.stats.outliers_influence import variance_inflation_factor
import statsmodels.api as sm
import shap

In [ ]:
origin = load_data("sk-diabetes")
origin.head()

### 📝 [3] 랜덤시드 고정

## 📘 #02. 탐색적 데이터 분석

### 📝 [1] 데이터 품질 검사

In [ ]:
np.random.seed(52)
desc = origin.describe().T
num_cols = origin.select_dtypes(include=np.number).columns
for column in num_cols:
    skewness = origin[column].skew()
    if abs(skewness) < 0.5:
        strength = "week"
        log_transform = "not needed"
    elif abs(skewness) < 1:
        strength = "normal"
        log_transform = "recommended"
    else:
        strength = "strong"
        log_transform = "needed"
    desc.loc[column, "skewness"] = skewness
    desc.loc[column, "skewness_strength"] = strength
    desc.loc[column, "log_transform"] = log_transform
desc

### 📝 [2] bmi, s3, s4 로그변환

## 📘 #03. 데이터 전처리

### 📝 [1] 훈련/검증 데이터 분리

### 📝 [2] 다중 공선성 제거

In [ ]:
df = origin.copy()
df['bmi'] = np.log1p(df['bmi'])
df['s3'] = np.log1p(df['s3'])
df['s4'] = np.log1p(df['s4'])
df.head()

In [ ]:
yname = "target"
x = df.drop(columns=[yname])
y = df[yname]
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.25, random_state=52)
x_train.shape, x_test.shape, y_train.shape, y_test.shape

In [ ]:
while True:
    vifs = {}
    exog = sm.add_constant(x_train)
    for i, col in enumerate(x_train.columns, start=0):
        vif = variance_inflation_factor(exog.values, i + 1)
        vifs[col] = vif
    vdf = DataFrame(list(vifs.items()), columns=["Variable", "VIF"])
    vdf.sort_values("VIF", ascending=False, inplace=True)
    print("==== VIF 계산 완료 ====")
    display(vdf)
    max_vif = vdf.iloc[0]['VIF']
    if max_vif > 10:
        drop_var = vdf.iloc[0]['Variable']
        print(f"Dropping '{drop_var}' with VIF: {max_vif:.2f}")
        x_train = x_train.drop(drop_var, axis=1)
        x_test = x_test.drop(drop_var, axis=1)
    else:
        break
display(x_train.head())
display(x_test.head())

## 📘 #04. 신경망 모델 적합

이전 실습 코드와 동일

### 📝 [1] 하이퍼파라미터 튜닝 정의

### 📝 [2] 튜너 객체 생성

### 📝 [3] 하이퍼파라미터 튜닝 수행

### 📝 [4] 최종 모형 도출

In [ ]:
_, cols = x_train.shape
print(f"최종 선택된 변수 개수: {cols}")
def tf_build(hp) -> Sequential:
    model = Sequential()
    model.add(Input(shape=(cols,)))
    model.add(Dense(units=hp.Choice("units", values=[4, 8, 16, 32, 64]), activation="relu"))
    model.add(Dense(1, activation="linear"))
    model.compile(
        optimizer="adam",
        loss="mse",
        metrics=["mae", RootMeanSquaredError(name="rmse"), R2Score(name="r2")],
    )
    return model

In [ ]:
tuner = Hyperband(
    hypermodel=tf_build,
    objective="val_mae",
    max_epochs=10,
    factor=3,
    seed=52,
    directory="D:\\tensor_hyperband",
    project_name="tf_hyperband_%s" % dt.now().strftime("%Y%m%d%H%M%S"),
)
tuner

In [ ]:
%%time
tuner.search(
    x_train, y_train, epochs=10, batch_size=32, validation_data=(x_test, y_test)
)
best_hps = tuner.get_best_hyperparameters()
if not best_hps:
    raise ValueError("No best hyperparameters found.")
print(f"""best hyperparameters: {best_hps[0].values}""")

## 📘 #05 성능평가

이전 실습 코드와 동일

### 📝 [1] 성능평가 지표

### 📝 [2] 학습 과정 확인

### 📝 [3] Loss, RMSE 학습곡선

In [ ]:
model = tuner.hypermodel.build(best_hps[0])
result = model.fit(
    x_train, y_train,
    epochs=500,
    validation_data=(x_test, y_test),
    verbose=0,
    callbacks=[
        TqdmCallback(verbose=1),
        EarlyStopping(monitor='val_loss', patience=5, min_delta=0.001),
        ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=0, verbose=1)
    ]
)
result

In [ ]:
train_eval = model.evaluate(x_train, y_train, verbose=0, return_dict=True)
test_eval = model.evaluate(x_test, y_test, verbose=0, return_dict=True)
final_results = DataFrame([train_eval, test_eval])
final_results.insert(0, "Dataset", ["Train", "Test"])
final_results["RMSE_gap"] = None
final_results.loc[1, "RMSE_gap"] = final_results.loc[1, "rmse"] - final_results.loc[0, "rmse"]
final_results

In [ ]:
history_df = DataFrame(data=result.history)
history_df["epoch"] = history_df.index + 1
history_df.head()

In [ ]:
figsize = (1600 / 100, 600 / 100)
fig, ax = plt.subplots(1, 2, figsize=figsize, dpi=100)
fig.subplots_adjust(wspace=0.2, hspace=0.2)
sb.lineplot(data=history_df, x="epoch", y="loss", ax=ax[0], label="Train Loss")
sb.lineplot(data=history_df, x="epoch", y="val_loss", ax=ax[0], label="Validation Loss")
ax[0].set_xlabel("Epoch")
ax[0].set_ylabel("Loss")
ax[0].set_title("Training vs Validation Loss")
ax[0].grid(True, alpha=0.3)
sb.lineplot(data=history_df, x="epoch", y="rmse", ax=ax[1], label="Train RMSE")
sb.lineplot(data=history_df, x="epoch", y="val_rmse", ax=ax[1], label="Validation RMSE")
ax[1].set_xlabel("Epoch")
ax[1].set_ylabel("RMSE")
ax[1].set_title("Training vs Validation RMSE")
ax[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
plt.close()

## 📘 #06. 예측 결과 활용

이전 실습 코드와 비슷

### 📝 [1] 예측치 구하기

### 📝 [2] 결과 데이터 셋 구성

### 📝 [3] 관측치와 예측치 비교 시각화

In [ ]:
pred = model.predict(x_test, verbose=0)
pred[:5]

In [ ]:
kdf = DataFrame({
    'bmi': x_test['bmi'],
    '실제값': y_test,
    '예측값': pred.flatten()
})
kdf['오차'] = kdf['실제값']-kdf['예측값']
kdf.head()

In [ ]:
figsize = (1280 / 100, 720 / 100)
fig, ax = plt.subplots(1, 1, figsize=figsize, dpi=100)
sb.regplot(data=kdf, x='bmi', y='실제값', label='실제값')
sb.regplot(data=kdf, x='bmi', y='예측값', label='예측값')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_title("실제값 vs 예측값")
plt.tight_layout()
plt.show()
plt.close()

## 📘 #07. 결과 해석 (SHAP)

### 📝 [1] Background 데이터 설정

딥러닝은 background 샘플 필요함. 50~200개 권장

In [ ]:
x_df = x_train.copy()
background = x_df.sample(100, random_state=42)
background_np = background.values
x_np = x_df.values
x_np

### 📝 [2] SHAP 계산

### 📝 [5] SHAP 분석 결과 확인

#### ✏ SHAP Value 계산

In [ ]:
explainer = shap.GradientExplainer(model, background_np)
shap_values = explainer.shap_values(x_np)
if isinstance(shap_values, list):
    shap_values = shap_values[0]
if shap_values.ndim == 3:
    shap_values = shap_values[:, :, 0]
shap_df = DataFrame(shap_values, columns=x_df.columns, index=x_df.index)
shap_df.head()

In [ ]:
summary_df = DataFrame({
    "feature": shap_df.columns,
    "mean_abs_shap": shap_df.abs().mean().values,
    "mean_shap": shap_df.mean().values,
    "std_shap": shap_df.std().values,
})
summary_df["direction"] = np.where(
    summary_df["mean_shap"] > 0,
    "양(+) 경향",
    np.where(summary_df["mean_shap"] < 0, "음(-) 경향", "혼합/미약")
)
summary_df["cv"] = summary_df["std_shap"] / (summary_df["mean_abs_shap"] + 1e-9)
summary_df["variability"] = np.where(summary_df["cv"] < 1, "stable", "variable")
summary_df = summary_df.sort_values("mean_abs_shap", ascending=False).reset_index(drop=True)
total_importance = summary_df["mean_abs_shap"].sum()
summary_df["importance_ratio"] = summary_df["mean_abs_shap"] / total_importance
summary_df["importance_cumsum"] = summary_df["importance_ratio"].cumsum()
summary_df["is_important"] = np.where(summary_df["importance_cumsum"] <= 0.80, "core", "secondary")
summary_df

#### ✏ 결과 시각화

In [ ]:
shap.summary_plot(shap_values, x_df, show=False)
fig = plt.gcf()
fig.set_size_inches(1024 / 100, len(summary_df) * 60 / 100)
fig.set_dpi(150)
plt.xlabel("SHAP value")
plt.tight_layout()
plt.show()
plt.close()

#### ✏ 💡 인사이트

- 🎯 핵심 예측변수 5개가 전체 중요도의 78.3% 차지
- 📊 변수 영향 방향의 차이 발견
- 🔄 모든 변수의 불안정한 영향력 유지
- 📈 변수 중요도 절대값 증가
- 🔀 변수 중요도 순위 변화
- 🎨 SHAP 분포도 패턴 변화